Connect to snowflake.

"In Databricks I built three reusable MERGE templates — SCD Type 1, SCD Type 2, and FACT upsert — each parameterized on table name and column lists. The SCD1 template handled three dim tables with zero code changes between them, just config differences. The FACT template performs 4-way LEFT JOINs to look up surrogate keys from each dim before MERGE, ensuring fact rows correctly reference the dim version active at sale time. I used the Snowflake Python connector for arbitrary DDL/DML rather than the Spark connector's JVM helper because the Python connector works across all cluster security modes — important for portability."

This whole episode is a stronger interview story than a clean run would have been:

"I caught a SCD2 violation in testing where 500 customers had two current rows. Root cause: parquet reader was wildcarding all date partitions, so staging contained both the customer's old version and new version during delta runs. The Step B insert then created phantom current rows. Fix: added a QUALIFY ROW_NUMBER() dedup step on staging keyed on the natural key + max updated_at. The bug self-corrected on the next clean run because the function then detected and closed the phantom rows on hash mismatch. I cleaned up the residual phantom closed rows with a separate dedup query keyed on (customer_id, scd_hash). Two lessons: write assertion queries with expected counts, not just smoke tests; and design MERGE logic that's idempotent and self-correcting where possible."

In [0]:
# print(dbutils.secrets.listScopes())

In [0]:
# print(dbutils.secrets.list("snowflake"))

Setup helpers

In [0]:

# Reads from ADLS → MERGEs into Snowflake

import snowflake.connector
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DecimalType

# Cluster auth setup — MUST run before any ADLS read

STORAGE_ACCOUNT = "storageaccount12344325"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net",
    dbutils.secrets.get(scope="snowflake", key="storage-account-key")
)

# Snowflake connection options (reusable)

sf_options = {
    "sfURL"      : dbutils.secrets.get("snowflake", "sf-url"),
    "sfUser"     : dbutils.secrets.get("snowflake", "sf-user"),
    "sfPassword" : dbutils.secrets.get("snowflake", "sf-password"),
    "sfDatabase" : "PHARMA_DB",
    "sfSchema"   : "CORE",
    "sfWarehouse": "AMIT_WAREHOUSE",
    "sfRole"     : "SYSADMIN"
}

# ADLS root
ADLS_ROOT = f"abfss://rawdata@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Helper functions
def read_all_parquet_from_adls(folder_path: str):
    """
    Reads ALL parquet files across ALL date partitions for the given folder.
    
    NOTE: This intentionally wildcards year/month/day to handle the case where
    a delta run lands new files alongside the initial load. Each MERGE template
    is responsible for deduplicating staging by primary key + latest updated_at
    before applying changes (see Step 0 in merge_scd2_customers and the dedup
    logic in merge_scd1 / merge_fact_sales).
    
    Why not "read latest partition only"?
      - Delta runs at midnight could split data across two date partitions
      - Per-template dedup by updated_at is more robust than path inference
    """
    full_path = f"{ADLS_ROOT}/{folder_path}/year=*/month=*/day=*/*.parquet"
    df = spark.read.parquet(full_path)
    print(f"  Read {df.count():,} rows from {folder_path}")
    return df


def write_to_snowflake_staging(df, stg_table: str):
    (df.write
       .format("net.snowflake.spark.snowflake")
       .options(**sf_options)
       .option("sfSchema", "RAW")
       .option("dbtable", stg_table)
       .mode("overwrite")
       .save())
    print(f"  Loaded {stg_table} (overwrite)")




def execute_snowflake_sql(sql: str):
    """
    Execute arbitrary SQL against Snowflake using the native Python connector.
    More portable than spark connector's JVM Utils.runQuery (which is blocked
    on Unity Catalog / table-ACL clusters).
    """
    conn = snowflake.connector.connect(
        user      = dbutils.secrets.get("snowflake", "sf-user"),
        password  = dbutils.secrets.get("snowflake", "sf-password"),
        account   = dbutils.secrets.get("snowflake", "sf-account"),
        warehouse = "AMIT_WAREHOUSE",
        database  = "PHARMA_DB",
        schema    = "CORE",
        role      = "SYSADMIN"
    )
    try:
        cur = conn.cursor()
        cur.execute(sql)
        # Try to fetch results — DDL/DML may not return rows, that's fine
        try:
            result = cur.fetchall()
        except snowflake.connector.errors.NotSupportedError:
            result = None
        cur.close()
        print(f"  Executed: {sql[:80]}...")
        return result
    finally:
        conn.close()

The SCD1 MERGE template:

In [0]:
def merge_scd1(source_table: str, target_table: str, primary_key: str, tracked_columns: list):
    """
    SCD Type 1 MERGE template — overwrite latest values + handle soft deletes.
    
    Flow:
      1. Read latest parquet from ADLS for `source_table`
      2. Load it into Snowflake STG_<source_table> (overwrite)
      3. Dedupe staging — keep latest row per primary key (handles multi-partition reads)
      4. MERGE STG → DIM_<target_table>:
         - WHEN MATCHED → UPDATE all tracked columns + is_deleted
         - WHEN NOT MATCHED → INSERT new row
    
    Args:
      source_table: e.g. 'products' (matches MySQL table name)
      target_table: e.g. 'DIM_PRODUCT' (Snowflake CORE table)
      primary_key:  e.g. 'product_id' (natural key, not surrogate)
      tracked_columns: list of cols to UPDATE on MATCH (excludes PK + audit cols)
    """
    print(f"\n[SCD1] Starting MERGE for {source_table} → {target_table}")
    
    # 1. Folder path follows ADLS convention: dim/<table_name>/
    folder = f"dim/{source_table}"
    
    # 2. Read raw parquet → DataFrame
    df = read_all_parquet_from_adls(folder)
    
    # 3. Write to Snowflake staging
    stg_table = f"STG_{source_table.upper()}"
    write_to_snowflake_staging(df, stg_table)
    
    # 4. Dedupe staging — keep latest row per primary key
    #    Reason: read_all_parquet_from_adls reads all date partitions, so staging
    #    may contain both initial-load and delta-load rows for the same key.
    dedupe_sql = f"""
    CREATE OR REPLACE TABLE PHARMA_DB.RAW.{stg_table} AS
    SELECT * FROM PHARMA_DB.RAW.{stg_table}
    QUALIFY ROW_NUMBER() OVER (PARTITION BY {primary_key} ORDER BY updated_at DESC) = 1
    """
    execute_snowflake_sql(dedupe_sql)
    print(f"  [SCD1] Step 0 complete: deduplicated staging by {primary_key}")
    
    # 5. Build the MERGE SQL dynamically based on tracked_columns
    update_clause = ",\n        ".join([f"t.{c} = s.{c}" for c in tracked_columns])
    insert_cols   = ", ".join([primary_key] + tracked_columns + ["is_deleted"])
    insert_vals   = ", ".join([f"s.{c}" for c in [primary_key] + tracked_columns + ["is_deleted"]])
    
    merge_sql = f"""
    MERGE INTO PHARMA_DB.CORE.{target_table} t
    USING PHARMA_DB.RAW.{stg_table} s
    ON t.{primary_key} = s.{primary_key}
    WHEN MATCHED THEN UPDATE SET
        {update_clause},
        t.is_deleted = s.is_deleted,
        t.etl_updated_at = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN INSERT ({insert_cols})
    VALUES ({insert_vals})
    """
    
    # 6. Execute MERGE on Snowflake
    execute_snowflake_sql(merge_sql)
    print(f"[SCD1] Completed {target_table}\n")

In [0]:
def merge_scd2_customers():
    """
    SCD Type 2 MERGE template for DIM_CUSTOMER.
    
    Logic:
      1. Read parquet from ADLS, write to STG_CUSTOMERS (overwrite)
      2. Deduplicate staging — keep only latest version per customer_id
         (handles case where read_all_parquet_from_adls reads multiple partitions)
      3. Two-step MERGE:
         a. Close old rows: any current row whose customer_id appears in staging
            with a DIFFERENT hash → set effective_to = now, is_current = false
         b. Insert new rows: customers in staging that either
            (i) don't exist in DIM_CUSTOMER yet, OR
            (ii) exist but their hash changed (closed in step a)
    
    Hash includes is_deleted so that soft-delete events trigger a new SCD2 version.
    """
    print("\n[SCD2] Starting MERGE for customers → DIM_CUSTOMER")
    
    # 1. Read & stage
    df = read_all_parquet_from_adls('dim/customers')
    write_to_snowflake_staging(df, 'STG_CUSTOMERS')
    
    # 2. Step 0 — deduplicate staging: keep latest version per customer_id
    dedupe_sql = """
    CREATE OR REPLACE TABLE PHARMA_DB.RAW.STG_CUSTOMERS AS
    SELECT * FROM PHARMA_DB.RAW.STG_CUSTOMERS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY updated_at DESC) = 1
    """
    execute_snowflake_sql(dedupe_sql)
    print("  [SCD2] Step 0 complete: deduplicated staging")
    
    # 3. Step A — close old rows whose attributes changed
    #    Hash includes is_deleted so soft-delete events trigger a new SCD2 version.
    close_old_sql = """
    UPDATE PHARMA_DB.CORE.DIM_CUSTOMER t
    SET effective_to   = CURRENT_TIMESTAMP(),
        is_current     = FALSE,
        etl_updated_at = CURRENT_TIMESTAMP()
    FROM PHARMA_DB.RAW.STG_CUSTOMERS s
    WHERE t.customer_id = s.customer_id
      AND t.is_current  = TRUE
      AND t.scd_hash   <> SHA2(CONCAT_WS('|',
                                  s.customer_name,
                                  s.customer_type,
                                  s.territory_id::STRING,
                                  COALESCE(s.address, ''),
                                  COALESCE(s.city, ''),
                                  COALESCE(s.state, ''),
                                  s.is_deleted::STRING
                              ), 256)
    """
    execute_snowflake_sql(close_old_sql)
    print("  [SCD2] Step A complete: closed changed rows")
    
    # 4. Step B — insert new versions (brand new customers + changed ones)
    insert_new_sql = """
    INSERT INTO PHARMA_DB.CORE.DIM_CUSTOMER (
        customer_id, customer_name, customer_type, territory_id,
        address, city, state, is_deleted,
        effective_from, effective_to, is_current, scd_hash
    )
    SELECT 
        s.customer_id,
        s.customer_name,
        s.customer_type,
        s.territory_id,
        s.address,
        s.city,
        s.state,
        s.is_deleted,
        CURRENT_TIMESTAMP()                AS effective_from,
        NULL                               AS effective_to,
        TRUE                               AS is_current,
        SHA2(CONCAT_WS('|',
            s.customer_name,
            s.customer_type,
            s.territory_id::STRING,
            COALESCE(s.address, ''),
            COALESCE(s.city, ''),
            COALESCE(s.state, ''),
            s.is_deleted::STRING
        ), 256)                            AS scd_hash
    FROM PHARMA_DB.RAW.STG_CUSTOMERS s
    WHERE NOT EXISTS (
        SELECT 1 
        FROM PHARMA_DB.CORE.DIM_CUSTOMER t
        WHERE t.customer_id = s.customer_id
          AND t.is_current  = TRUE
    )
    """
    execute_snowflake_sql(insert_new_sql)
    print("  [SCD2] Step B complete: inserted new versions")
    print("[SCD2] Completed DIM_CUSTOMER\n")

In [0]:
def apply_fact_transformations(df):
    """
Apply PySpark DataFrame transformations to fact data before MERGE:
  - Cast transaction_date timestamp → DateType
  - Cast unit_price to decimal(10,2) to match source/target precision
  - Recompute total_amount = quantity × unit_price (defensive consistency)
  - Derived analytics columns: day_of_week, month_year, revenue_tier
  - ETL audit timestamp (transformed_at)
  - Row-count assertion to catch silent data loss
"""
    input_count = df.count()
    print(f"  [transform] Input rows: {input_count:,}")
    
    transformed = (df
        .withColumn("transaction_date", F.col("transaction_date").cast(DateType()))
        .withColumn("unit_price",       F.col("unit_price").cast(DecimalType(10, 2)))
        .withColumn("total_amount",
            (F.col("quantity") * F.col("unit_price")).cast(DecimalType(12, 2)))
        .withColumn("day_of_week", F.dayofweek(F.col("transaction_date")))
        .withColumn("month_year",  F.date_format(F.col("transaction_date"), "yyyy-MM"))
        .withColumn("revenue_tier",
            F.when(F.col("total_amount") < 1000,  "Low")
             .when(F.col("total_amount") < 10000, "Mid")
             .otherwise("High"))
        .withColumn("transformed_at", F.current_timestamp())
    )
    
    print("  [transform] Revenue tier distribution:")
    transformed.groupBy("revenue_tier").count().show()
    
    output_count = transformed.count()
    print(f"  [transform] Output rows: {output_count:,}")
    assert input_count == output_count, (
        f"Row count mismatch! Input {input_count:,} != Output {output_count:,}"
    )
    print(f"  [transform] ✓ Row count preserved")
    
    return transformed

def merge_fact_sales():
    """
    FACT MERGE template — PySpark transforms + surrogate key joins + late-arriving dim handling.
    
    Flow:
      1. Read all parquet from ADLS for fact/sales
      2. Apply PySpark transformations (apply_fact_transformations):
         - Cast types (date, decimal precision)
         - Defensive recompute of total_amount
         - Derived analytics columns (day_of_week, month_year, revenue_tier)
         - Row-count assertion
      3. Load to STG_SALES_TRANSACTIONS (overwrite)
      4. Dedupe staging by transaction_id — keep latest row per transaction
         (handles multi-partition reads from read_all_parquet_from_adls)
      5. MERGE with 4-way LEFT JOIN to dims for surrogate key lookup:
         - WHEN MATCHED + soft delete → mark deleted
         - WHEN MATCHED + customer_sk NULL → backfill (late-arriving SCD2 dim)
         - WHEN NOT MATCHED → INSERT new transaction (populates derived cols too)
    
    Design note: facts are immutable except for soft-delete and late-arriving dim
    backfills. Real-world sales transactions are corrected via reversal entries,
    not row updates — that's standard accounting practice (audit trail).
    
    Derived analytics columns are populated only on INSERT (point-in-time values).
    They are NOT recomputed on update operations — keeps audit trail clean.
    """
    print("\n[FACT] Starting MERGE for sales_transactions → FACT_SALES")
    
    # 1. Read raw parquet
    df = read_all_parquet_from_adls('fact/sales')
    
    # 2. Apply PySpark transformations (type cleanup + derived columns + assertion)
    df = apply_fact_transformations(df)

    # 3. Load to staging
    write_to_snowflake_staging(df, 'STG_SALES_TRANSACTIONS')
    
    # 4. Step 0 — deduplicate staging by transaction_id
    #    Reason: read_all_parquet_from_adls reads all date partitions, so staging
    #    may contain duplicate transaction_ids if a transaction was updated.
    dedupe_sql = """
    CREATE OR REPLACE TABLE PHARMA_DB.RAW.STG_SALES_TRANSACTIONS AS
    SELECT * FROM PHARMA_DB.RAW.STG_SALES_TRANSACTIONS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY updated_at DESC) = 1
    """
    execute_snowflake_sql(dedupe_sql)
    print("  [FACT] Step 0 complete: deduplicated staging by transaction_id")
    
    # 5. MERGE with surrogate key lookups
    
    merge_sql = """
MERGE INTO PHARMA_DB.CORE.FACT_SALES t
USING (
    SELECT 
        s.transaction_id,
        dp.product_sk,
        dc.customer_sk,
        dr.rep_sk,
        dt.territory_sk,
        s.transaction_date,
        s.quantity,
        s.unit_price,
        s.total_amount,
        s.is_deleted,
        s.day_of_week,
        s.month_year,
        s.revenue_tier,
        s.transformed_at
    FROM PHARMA_DB.RAW.STG_SALES_TRANSACTIONS s
    LEFT JOIN PHARMA_DB.CORE.DIM_PRODUCT    dp ON dp.product_id   = s.product_id
    LEFT JOIN PHARMA_DB.CORE.DIM_CUSTOMER   dc ON dc.customer_id  = s.customer_id
                                              AND dc.is_current   = TRUE
    LEFT JOIN PHARMA_DB.CORE.DIM_SALES_REP  dr ON dr.rep_id       = s.rep_id
    LEFT JOIN PHARMA_DB.CORE.DIM_TERRITORY  dt ON dt.territory_id = dr.territory_id
) src
ON t.transaction_id = src.transaction_id

WHEN MATCHED AND src.is_deleted = TRUE THEN UPDATE SET 
    t.is_deleted = TRUE,
    t.etl_loaded_at = CURRENT_TIMESTAMP()

WHEN MATCHED AND t.customer_sk IS NULL AND src.customer_sk IS NOT NULL THEN UPDATE SET
    t.customer_sk = src.customer_sk,
    t.etl_loaded_at = CURRENT_TIMESTAMP()

WHEN NOT MATCHED THEN INSERT (
    transaction_id, product_sk, customer_sk, rep_sk, territory_sk,
    transaction_date, quantity, unit_price, total_amount, is_deleted,
    day_of_week, month_year, revenue_tier, transformed_at
)
VALUES (
    src.transaction_id, src.product_sk, src.customer_sk, src.rep_sk, src.territory_sk,
    src.transaction_date, src.quantity, src.unit_price, src.total_amount, src.is_deleted,
    src.day_of_week, src.month_year, src.revenue_tier, src.transformed_at
)
"""
    execute_snowflake_sql(merge_sql)
    print("[FACT] Completed FACT_SALES\n")

In [0]:

# DISPATCHER — runs all 5 MERGEs in correct order
# This is the cell ADF will trigger

print("=" * 60)
print("Pipeline run started")
print("=" * 60)

# 1. SCD1 dims (3 tables — order doesn't matter among these)
merge_scd1(
    source_table='territories',
    target_table='DIM_TERRITORY',
    primary_key='territory_id',
    tracked_columns=['territory_name', 'region', 'country']
)

merge_scd1(
    source_table='products',
    target_table='DIM_PRODUCT',
    primary_key='product_id',
    tracked_columns=['product_name', 'category', 'manufacturer', 'unit_price']
)

merge_scd1(
    source_table='sales_reps',
    target_table='DIM_SALES_REP',
    primary_key='rep_id',
    tracked_columns=['rep_name', 'email', 'territory_id', 'hire_date']
)

# 2. SCD2 customer (must run after SCD1, before FACT)
merge_scd2_customers()

# 3. FACT (must run LAST — depends on all dims being populated)
merge_fact_sales()

print("=" * 60)
print("✅ Pipeline run complete")
print("=" * 60)

In [0]:
verify_sql = """
SELECT 'DIM_TERRITORY'   AS tbl, COUNT(*) AS cnt FROM PHARMA_DB.CORE.DIM_TERRITORY
UNION ALL
SELECT 'DIM_PRODUCT',            COUNT(*)        FROM PHARMA_DB.CORE.DIM_PRODUCT
UNION ALL
SELECT 'DIM_SALES_REP',          COUNT(*)        FROM PHARMA_DB.CORE.DIM_SALES_REP
UNION ALL
SELECT 'DIM_CUSTOMER (total)',   COUNT(*)        FROM PHARMA_DB.CORE.DIM_CUSTOMER
UNION ALL
SELECT 'DIM_CUSTOMER (current)', COUNT(*)        FROM PHARMA_DB.CORE.DIM_CUSTOMER WHERE is_current = TRUE
UNION ALL
SELECT 'FACT_SALES',             COUNT(*)        FROM PHARMA_DB.CORE.FACT_SALES
"""
print(execute_snowflake_sql(verify_sql))